# Score the eight datasets

Scores the context mean, PIE and PWPA with PIE's metric code (FDR 0.05) on every development fold, then takes the mean and the standard error over folds.
The context mean predicts each context's mean training knockdown: its expression change, its fold change, and each gene's DE frequency.

PIE's metric code comes from the `pie` package ([ArcInstitute/pie](https://github.com/ArcInstitute/pie) at `21a526a`), so run this notebook in PIE's environment or with PIE's `src/` on `PYTHONPATH`.

Writes `results/eight_datasets_scores.tsv` and draws `figures/eight_datasets.png`.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir
from pie_scoring import load, pie_metrics, read_predictions, split

In [ ]:
datasets = (
    pd.read_csv("../0.datasets/datasets.tsv", sep="\t")
    .query("used_in == '2.train_test' and dataset != 'pie_replogle_nadig'")
    .dataset.tolist()
)

rows = []
for dataset in datasets:
    d = load(data_dir("perturbseq", "processed", dataset, "preprocessed"))
    splits = pd.read_csv(f"../0.datasets/splits/{dataset}.tsv", sep="\t")
    for fold in range(4):
        s = split(splits, fold)
        test, train = np.flatnonzero(np.isin(d.pert, s["test"])), np.isin(d.pert, s["train"])
        mean = {}
        for c in np.unique(d.context):
            r = train & (d.context == c)
            mean[c] = (d.de[r].mean(0), np.nan_to_num(np.nanmean(d.lfc[r], 0)), d.delta[r].mean(0, dtype=np.float64))
        predictions = {"context_mean": (test, *(np.stack([mean[c][i] for c in d.context[test]]) for i in range(3)))}
        for model in ("pie", "pwpa"):
            predictions[model] = read_predictions(data_dir("predictions", model, dataset, f"fold_{fold}") / f"{model}_f{fold}.parquet", d)
            assert sorted(predictions[model][0]) == sorted(test), f"{model} {dataset} fold {fold}: not the split's test rows"
        for model, p in predictions.items():
            rows += [{"dataset": dataset, "model": model, "fold": fold, "metric": m, "value": v} for m, v in pie_metrics(d, *p).items()]

g = pd.DataFrame(rows).groupby(["dataset", "model", "metric"], sort=False).value
scores = pd.DataFrame({"mean": g.mean(), "se": g.std(ddof=1) / np.sqrt(g.count()), "n_folds": g.size()}).reset_index()
scores.to_csv("results/eight_datasets_scores.tsv", sep="\t", index=False)

In [ ]:
import matplotlib.pyplot as plt

scores = pd.read_csv("results/eight_datasets_scores.tsv", sep="\t")
DATASETS = {
    "replogle_k562": "Replogle 2022, K562",
    "replogle_rpe1": "Replogle 2022, RPE1",
    "nadig_hepg2": "Nadig 2025, HepG2",
    "nadig_jurkat": "Nadig 2025, Jurkat",
    "kolf_ipsc": "Nourreddine 2026, KOLF2.1J ‡",
    "vcc_h1": "Roohani 2025, H1 (VCC)",
    "shi_brain": "Shi 2026, mouse brain *",
    "geiger_bmdc": "Geiger-Schuller 2026, mouse DCs †",
}
METRICS = {
    "binary_auprc": "AUPRC",
    "sig_jaccard": "Jaccard",
    "direction_match": "DE direction\nmatch",
    "spearman_nsig": "Spearman\neffect size",
    "spearman_lfc": "Spearman\nLFC",
    "discrimination_score_l1": "Discrimination\nL1",
}
MODELS = {"context_mean": "Context mean", "pie": "PIE", "pwpa": "PWPA"}

# per dataset: rank the models on each metric (1 = best, ties share a rank, on the 3 decimals shown),
# then average over the metrics a model has; the context mean has no Spearman effect size
table = scores.pivot_table(index=["dataset", "model"], columns="metric", values=["mean", "se"])
shown = table["mean"].round(3)
rank = shown.groupby(level="dataset").rank(ascending=False, method="min")
avg_rank = rank.mean(axis=1)
best = shown.eq(shown.groupby(level="dataset").transform("max"))
overall = avg_rank.eq(avg_rank.groupby(level="dataset").transform("min"))

ROW, X = 0.3, [0.0, 2.45, 3.6, 4.8, 6.0, 7.2, 8.4, 9.6, 10.85]  # row height (in), column left edges
fig = plt.figure(figsize=(11.6, ROW * (len(DATASETS) * len(MODELS) + 4)))
ax = fig.add_axes((0, 0, 1, 1))
ax.set_xlim(-0.1, 11.5)
ax.set_ylim(len(DATASETS) * len(MODELS) + 3.6, -0.4)
ax.axis("off")
rule = lambda y, lw: ax.plot([-0.05, 11.45], [y, y], color="black", lw=lw, solid_capstyle="butt")  # noqa: E731

for x, head in zip(X, ["Dataset", "Model", *METRICS.values(), "Avg.\nrank ↓"]):
    ax.text(x, 0.5, head, va="center", fontweight="bold", fontsize=9.5, linespacing=1.1)
rule(-0.2, 1.4)
rule(1.2, 0.8)
y = 0.7
for dataset, label in DATASETS.items():
    for i, (model, name) in enumerate(MODELS.items()):
        y += 1
        key = (dataset, model)
        if i == 0:
            ax.text(X[0], y, label, va="center", fontsize=9)
        ax.text(X[1], y, name, va="center", fontsize=9, fontweight="bold" if overall[key] else "normal")
        for x, metric in zip(X[2:], METRICS):
            mean, se = table.loc[key, ("mean", metric)], table.loc[key, ("se", metric)]
            text = "–" if pd.isna(mean) else f"{mean:.3f} ± {se:.3f}"
            ax.text(x, y, text, va="center", fontsize=9, fontweight="bold" if best.loc[key, metric] else "normal")
        ax.text(X[-1] + 0.15, y, f"{avg_rank[key]:.1f}", va="center", fontsize=9, fontweight="bold" if overall[key] else "normal")
    rule(y + 0.5, 1.4 if dataset == list(DATASETS)[-1] else 0.4)
ax.text(X[0], y + 1.0, "Mean ± standard error over development folds 0–3. Bold: the best model for each dataset and metric, "
        "and the best average rank over metrics.\nThe context mean has no Spearman effect size, so its rank averages the other "
        "five metrics. Footnotes ‡ * † are in 2.train_test/README.md.", va="top", fontsize=8, color="#444444")
fig.savefig("figures/eight_datasets.png", dpi=200, facecolor="white")

\* Shi: in vivo data from 74 mice. The labels use PIE's cell-level DE, which treats each cell as a replicate and ignores mouse-to-mouse variation.

† Geiger-Schuller: primary mouse dendritic cells in 16 hashtagged samples over 2 rounds. Cell-level DE ignores this sample structure; a pseudobulk test across samples may label differently.

‡ Nourreddine (KOLF): each knockdown was screened in one of three batches, but cell-level DE compares it with non-targeting cells pooled across all batches (PIE's default), so batch differences can enter the DE labels.